# View saved outputs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RollinsE/open-bandit-ab-testing/blob/main/notebooks/01_view_saved_outputs.ipynb)

Open a finished build or the promoted release without re-running anything.

In [ ]:
# Setup: clone the repository (Colab) or use the local checkout, then install.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/RollinsE/open-bandit-ab-testing.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = Path("/content/open-bandit-ab-testing")
    if not PROJECT_DIR.exists():
        !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
    elif (PROJECT_DIR / ".git").exists():
        !git -C {PROJECT_DIR} pull --ff-only -q  # pick up any updates in a session that is still running
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
%pip install -q -r requirements/colab.txt
print("Project folder:", PROJECT_DIR)

In [ ]:
import json
import pandas as pd
from IPython.display import HTML, display

USE_DRIVE = False  # True if the run notebook kept outputs on Google Drive
OUTPUT_DIR = "/content/outputs" if IN_COLAB else "outputs"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/open-bandit/outputs"
# Point this at a build folder, or leave as the promoted release.
TARGET = Path(OUTPUT_DIR) / "releases" / "latest"
if not TARGET.exists():
    TARGET = sorted(Path(OUTPUT_DIR, "builds").iterdir())[-1]
print("Showing:", TARGET)
print(json.dumps(json.loads((TARGET / "run_manifest.json").read_text()), indent=2))

In [ ]:
for name in sorted((TARGET / "artifacts").glob("*.csv")):
    print(f"\n== {name.name}")
    display(pd.read_csv(name).head(10))

In [ ]:
# Interactive Plotly figures
for fig in sorted((TARGET / "reports" / "figures").glob("*.html")):
    print(fig.stem)
    display(HTML(fig.read_text()))